# 03 — Colab single-image debug

Mirrors `02_inference.ipynb` but for iteration: load the model **once**, then re-run the pipeline cell as many times as you want with different configs / prompts / images. ~2 min cold start, then ~30s per image.

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier works for 4-bit 7B).

**Optional:** to load a LoRA adapter from HF Hub, add an `HF_TOKEN` secret via the key icon in the left sidebar (only needed for private repos).

In [ ]:
# 1. Install GPU stack. Colab already has torch + CUDA.
%pip -q install --upgrade unsloth unsloth_zoo
%pip -q install --no-deps trl peft accelerate bitsandbytes
%pip -q install pyyaml opencv-python-headless

In [ ]:
GITHUB_REPO     = "https://github.com/dhodyrev/handwritten-to-data.git"
HF_ADAPTER_REPO = "dkhodyriev/htr-qwen25vl-transcribe-lora"  # set None for base model
CONFIG          = "configs/pipeline.yaml"                     # or configs/pipeline_p1.yaml

In [ ]:
# 2. Clone repo and install htr.
import os, subprocess, sys
REPO_DIR = "/content/repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], check=True)
print("cwd:", os.getcwd())

In [ ]:
# 3. Optional HF login (only needed if the adapter repo is private).
if HF_ADAPTER_REPO:
    try:
        from google.colab import userdata
        from huggingface_hub import login
        login(token=userdata.get("HF_TOKEN"))
        print("hf login OK")
    except Exception as e:
        print(f"hf login skipped ({e}); proceeding anonymously (only works if adapter is public)")

In [ ]:
# 4. Load model ONCE — keep this cell's variables alive while iterating below.
from htr.backend import DEFAULT_MODEL, load_model
handle = load_model(DEFAULT_MODEL, adapter_path=HF_ADAPTER_REPO)
print("model ready:", handle.name, "+ adapter" if handle.adapter_path else "(base)")

In [ ]:
# 5. Get one image. Pick A or B.
#
# A) Upload from your Mac:
import os
from google.colab import files
uploaded = files.upload()                          # opens file picker
# files.upload() saves into the *current* working dir (cell 3 chdir'd to the
# repo), so resolve the real path instead of assuming /content/.
IMAGE_PATH = os.path.abspath(next(iter(uploaded)))
SOURCE = "unknown"                                 # or school / university / dictation / archive

# B) Or pull a sample page from the HF dataset (uncomment, comment out A):
# from htr.data import materialize_image
# IMAGE_PATH = materialize_image("train", "some_page_filename.jpg")
# SOURCE = "school"

print("image:", IMAGE_PATH)

In [ ]:
# 6. Run the pipeline. Re-run this cell after tweaking CONFIG / SOURCE / prompts.
import importlib, htr.pipeline, htr.prompts
importlib.reload(htr.prompts); importlib.reload(htr.pipeline)  # pick up prompt edits without restart
from htr.config import load_pipeline_config
from htr.pipeline import run_pipeline
from pathlib import Path

cfg = load_pipeline_config(CONFIG)
result = run_pipeline(IMAGE_PATH, Path(IMAGE_PATH).stem, SOURCE, cfg)

print(f"{len(result['regions'])} regions\n")
for i, r in enumerate(result["regions"]):
    print(f"[{i}] {r.get('type','?'):12s} bbox={r['bbox']}")
    print(f"     {r.get('transcription','')[:200]}")
    print()

In [ ]:
# 7. Visualise: draw bboxes over the image.
from PIL import Image, ImageDraw
img = Image.open(IMAGE_PATH).convert("RGB")
W, H = img.size
draw = ImageDraw.Draw(img)
for i, r in enumerate(result["regions"]):
    x0, y0, x1, y1 = r["bbox"]
    # bboxes are normalised to 0–1000
    box = [x0 * W / 1000, y0 * H / 1000, x1 * W / 1000, y1 * H / 1000]
    draw.rectangle(box, outline="red", width=3)
    draw.text((box[0] + 4, box[1] + 4), str(i), fill="red")
img.thumbnail((1200, 1200))
img

## Iterating

- **Prompt tweak:** edit `src/htr/prompts.py` in the file browser, then re-run cell 6 (the `importlib.reload` line picks it up — no kernel restart).
- **Config tweak:** edit `configs/pipeline.yaml` (or switch `CONFIG` to `pipeline_p1.yaml`), re-run cell 6.
- **Different image:** re-run cell 5 (option A re-uploads), then cell 6.
- **Different source routing:** change `SOURCE` in cell 5, re-run cell 6.

Model stays loaded in cell 4's `handle` — only restart that cell if you change `HF_ADAPTER_REPO` or the base model.